# 01. Dataset and Mask Visualization

Inspects `dataset_manifest.json`, loads real/synthetic speech bubble dataset samples, validates image and mask spatial dimensions, confirms binary mask value ranges `{0, 255}`, and renders side-by-side grids with transparent overlays.


In [ ]:
import json
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

manifest_path = "../data/processed/real/dataset_manifest.json"
if not os.path.exists(manifest_path):
    manifest_path = "../data/processed/synthetic/synthetic_manifest.json"

print(f"Loading manifest from: {manifest_path}")
if os.path.exists(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(f"Source: {manifest.get('dataset_source', 'Synthetic')}")
    print(f"Total Samples: {manifest.get('sample_count', 0)}")
else:
    print("Manifest not found. Run download_dataset.py first.")



In [ ]:
data_dir = os.path.dirname(manifest_path)
images_dir = os.path.join(data_dir, "images")
masks_dir = os.path.join(data_dir, "masks")

img_paths = sorted(glob.glob(os.path.join(images_dir, "*.png")))[:6]
print(f"Found {len(img_paths)} sample images to inspect.")

for img_p in img_paths:
    base = os.path.basename(img_p)
    mask_p = os.path.join(masks_dir, base)
    assert os.path.exists(mask_p), f"Missing corresponding mask for {base}"
    
    img = np.array(Image.open(img_p))
    mask = np.array(Image.open(mask_p))
    
    # Assert dimension alignment
    assert img.shape[:2] == mask.shape[:2], f"Dimension mismatch: {img.shape} vs {mask.shape}"
    # Assert mask is binary
    unique_vals = set(np.unique(mask))
    assert unique_vals.issubset({0, 255}), f"Non-binary values in mask: {unique_vals}"
print("Dimension alignment and binary mask value validation PASSED!")



In [ ]:
fig, axes = plt.subplots(len(img_paths), 3, figsize=(12, 4 * max(1, len(img_paths))))
if len(img_paths) == 1:
    axes = np.expand_dims(axes, 0)

for i, img_p in enumerate(img_paths):
    base = os.path.basename(img_p)
    mask_p = os.path.join(masks_dir, base)
    img = np.array(Image.open(img_p))
    mask = np.array(Image.open(mask_p))
    
    # Original image
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"{base} - Image")
    axes[i, 0].axis("off")
    
    # Binary mask
    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title("Binary Mask (0=bg, 255=bubble)")
    axes[i, 1].axis("off")
    
    # Red transparent overlay
    overlay = img.copy()
    overlay[mask > 127] = (overlay[mask > 127] * 0.4 + np.array([255, 0, 0]) * 0.6).astype(np.uint8)
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Segmentation Overlay")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

